# Text mining using LLM and Retrieval Augmented Generation (RAG) 

In this last exercise of the block course, we will extract information about impacts of natural hazards, such as floods or heatwaves, on critical infrastructure. For example, the far-reaching societal and economical impacts of the recent power black in Berlin some months ago is an example. The blackout highlighted what kind of impacts we can experience in our daily life, but also on the economy, when some of the critical infrastructure is not operational anymore. However, of course the recent blackout was caused by human actions and not by a natural force. 

In detail, we focus on the extraction of direct impacts from natural hazards, e.g. disruptions and closures in the transport network, power blackouts, partly operational wastewater treatment plants. This means that we do not extract indirect impacts of disrupted critical infrastructure, such as cascading impacts caused by a power blackout such as loss of telecommunication/IT infrastructure, societal and economic impacts.  



## Goal of the exercise: 
We aim to compare different LLMs in regard to their ability to extract information about disrupted critical infrastructure from newspapers and scientific publications. Each LLM should extract textual information and compress it into tabular data. A simple .csv file will be our output with columns about the affected infrastructure, its damage type, and location, including also the reference from where the information was extracted - namely document name and the text snippets which was used by the LLM to extract the information.\

The two different LLM workflows differ in their complexity, hereby we keep the model itself simple (e.g. Llama 2 trained on 7 billion parameters) but rather extend the workflow around it by using techniques of prompt engineering and guidance of the LLM with rule-based relationship extraction.

> NOTE: We dont conduct any model training of the LLM - for the respective task it would be time consuming and would quite a lot of computational resources 

## Data download

You can use simply the data (zip folder) directly located in the folder where also the notebook is placed. Unzip the data folder and place it in a folder called `./data` in the same directory of this exercise.


## Load packages and some settings

In [14]:
# In case ypu Have GPU and CUDA installed, then these settings might be useful  

%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=0  # nvidia gpu
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
# %env TORCH_USE_CUDA_DSA=0


env: CUDA_DEVICE_ORDER=PCI_BUS_ID
env: CUDA_VISIBLE_DEVICES=0  # nvidia gpu
env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


In [15]:
import os

from glob import glob
from pathlib import Path

from io import StringIO

import pandas as pd
from jinja2 import  Environment, FileSystemLoader
from langchain_docling import DoclingLoader

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
import torch
import gc  # cleaning up

from settings import settings as s
import src.data_handler as d


#  automatic linebreaks and multi-line cells.
pd.set_option("display.max_colwidth", None)
pd.set_option("display.colheader_justify", "left")

# speed up LLM inference
batch_size_gpu = 8  # max for my nvidia GPU
# play around to find best size to better utilize the GPU (e.g. check via nvtop) 


## check that cuda is ready
torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)



# load data paths
DATA_DIR = s.PATH_DATA
LLM_OUTPUT_DIR = s.PATH_LLM_DATA



0.0


### Load data

In [16]:
# Lets check how our input data looks like
text_sources = d.load_text_sources(DATA_DIR)
print(text_sources)

['../data/Fekete 2025 - Cascading impact chains and recovery challenges of the 2024 Valencia catastrophic floods_cleaned.md', '../data/Kettle 2020 - Storm Xaver over Europe in December 2013 Overview of energy impacts and North Sea events_cleaned.md', '../data/Treanor 2015 - Storm Desmond damage across Cumbria estimated at £500m _ Storm Desmond _The Guardian_cleaned.md', '../data/Skoulding 2023 - Where are the fires in Italy today as temperatures rise to 47.6C on Sicily_ _ The Independent_cleaned.md', '../data/McGregor 2007 - The social impacts of heat waves_cleaned.md', '../data/PWC 2015 - Updated estimates on cost of Storm Desmond_cleaned.md', '../data/The Guardian 2015 - Storm Desmond damage across Cumbria estimated at £500m _ Storm Desmond_cleaned.md', '../data/ABC 2024 - Traffic jams and flight delays due to heavy rain and lightning storm in Malaga_cleaned.md', '../data/Koks 2019 - Understanding Business Disruption and Economic Losses Due to Electricity Failures and Flooding_cleane

### Example test (with pytest) 

For a more thorough introduction to code testing, have a look for example [here](https://realpython.com/pytest-python-testing/#what-makes-pytest-so-useful)

Keep in mind that tests usually follow the Arrange-Act-Assert model:
* Arrange, or set up, the conditions for the test
* Act by calling some function or method
* Assert that some end condition is true


In [17]:
# In a notebook we use a variation of pytest, called ipytest, to run our tests and to plot the test results directly in the notebook. 
import ipytest
ipytest.autoconfig()
import tempfile



In [18]:
%%ipytest

# write pytest for this function
def test_load_text_sources():

    # create a temporary directory and files for testing
    with tempfile.TemporaryDirectory() as tmpdirname:
        # create some test files
        test_files = [
            "file1_cleaned.md",
            "file2_cleaned.md",
            "file3.md",  # this should be ignored
            "file3_cleaned.txt",  # this should be ignored
        ]
        for filename in test_files:
            with open(os.path.join(tmpdirname, filename), "w") as f:
                f.write("test content")

        # call the function to test
        result = d.load_text_sources(tmpdirname)

        # check that only the correct files are returned
        expected_files = [
            os.path.join(tmpdirname, "file1_cleaned.md"),
            os.path.join(tmpdirname, "file2_cleaned.md"),
        ]
        assert set(result) == set(expected_files)

.                                                                                            [100%]
1 passed in 0.01s


## LLM v1 (first attempt )

This LLM just tries to extract information about hazard impacts across Europe - namely which infrastructure was affected by the hazard, how it was affected (its damage type or in-operability), and where this hazard impact happened (geolocation of the affected infrastructure). In other words, the LLM should return at least 3 columns (infrastructure type, damage type, and location).


Find out if you can use GPU and define the name of the LLM from Huggingface you want to use for text mining


In [19]:
device = transformers.infer_device()
print(f"Using device: {device}")


# Name the model and tokenizer we want to use for text mining
model_name = "meta-llama/Llama-2-7b-chat-hf"



Using device: cuda


### Prompt 1
The step comprises some prompt engineering by loading a prompt template with some prompt variables: user "question" and "context". The latter refers to the text chunks that are passed to the model iteratively while scanning through each document, the former to the question of the user (defined in the code block below)

In [20]:
question = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, and the type of damage."


In [21]:
env = Environment(loader=FileSystemLoader("./prompt_templates/"))
prompt_template = env.get_template("simple_prompt.txt")


Now we have the prompt template ready, but before passing it to the LLM we want to render it to make sure that the model recognizes that we have some dynamic data in the prompts (see the prompt variable `context). Without the variables in the prompt above, we would not need the rendering.

> NOTE that loading LLM prompts as templates and rendering them (inserting dynamic data into a structured format) is a fundamental best practice in AI application development. It simply make your code more flexible and robust. This approach moves beyond ad-hoc prompting to a more reusable and scalable prompt. 



In [22]:
# quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

In [23]:


## check that cuda is ready
torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)




0.0


In [25]:
print(f"Using {device}")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    attn_implementation="flash_attention_2", 
    quantization_config=bnb_config,
    local_files_only=True,  # <-- uncomment when already downloaded model once
)
# model.save_pretrained("./huggingface_mirror/hub/") 

tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    use_fast=True,
    local_files_only=True,  # <-- uncomment when already downloaded model once
)
# tokenizer.save_pretrained("./huggingface_mirror/hub/")


# For the people with GPU:
# It might also be benefical to reduce memory usage furhter by moving the model to the GPU and enabling checkpointing. However the latter is only benefical when you want to do model training, however, for now we just use the (pre-trained) LLM for inference (ie. prediction)
model = model.to(device)
model.use_checkpointing = True  # <<-- uncomment when you want to reduce GPU memory usage during training

Using cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [27]:
## The entire extraction process looks like this 

df_responses_all = pd.DataFrame()


## NOTE: we limit the data extraction to only two publications at the moment to exemplify how the LLM application works
text_sources = glob(str(Path(DATA_DIR, "*_cleaned.md" ))) [:2]


## initialize pipeline
pipeline = transformers.pipeline(  # load model locally from .cache\
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
)

for i, filename in enumerate(text_sources):

    no_documents = len(text_sources)
    filepath = Path(filename)

    ## load document and break into chunks 
    loader = DoclingLoader(filepath) 
    doc = loader.load()


    print(f"\n\n ######## -------- Processing document [{i+1}/{no_documents}]: {filepath.name} -------- ######## \n")


    for j, chunk in enumerate(doc):

        print("chunk:", j)

        ## render prompt
        context = [{"text": chunk.page_content}]

        rendered_prompt = prompt_template.render(
            context=context,
            question=question,
        )

        # call pipeline on the respective chunk
        response = pipeline(
            text_inputs=rendered_prompt,
            max_new_tokens=1024, 
            do_sample=True,
            num_beams=1, 
            temperature=0.2,
            eos_token_id=tokenizer.eos_token_id,
            return_full_text=False, 
        )
        # postprocess response
        resp = response[0]["generated_text"].replace("\n", "")
        try:
            resp = pd.read_json(StringIO(resp))
            resp["filename"] = filepath.name  
            resp["chunk"] = chunk.page_content 
            df_responses_all = pd.concat([df_responses_all, resp], ignore_index=True)
            
        except ValueError as e:
            print(f"Cannot add response to output dataframe: {e}, \n{resp}")


    # empty CUDA cache for next document
    gc.collect()
    torch.cuda.empty_cache()






Device set to use cuda:0


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
Token indices sequence length is longer than the specified maximum sequence length for this model (552 > 512). Running this sequence through the model will result in indexing errors




 ######## -------- Processing document [1/2]: Fekete 2025 - Cascading impact chains and recovery challenges of the 2024 Valencia catastrophic floods_cleaned.md -------- ######## 

chunk: 0
Cannot add response to output dataframe: Trailing data, 
 [{"infrastructure_type": "airport","damage": "runway damaged","location": "Valencia"},{"infrastructure_type": "road","damage": "road damaged","location": "access road"},{"infrastructure_type": "railway","damage": "railway damaged","location": "railway infrastructure"},{"infrastructure_type": "harbor","damage": "harbor damaged","location": "port"},{"infrastructure_type": "electricity","damage": "power plant damaged","location": "power grid"},{"infrastructure_type": "water","damage": "water treatment plant damaged","location": "water supply"},{"infrastructure_type": "medical-care","damage": "hospital damaged","location": "healthcare facilities"}]Explanation:* The first case is an airport with a damaged runway.* The second case is a road with d

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


chunk: 10
Cannot add response to output dataframe: Expected object or value, 
 Here are the answers to the question based on the provided context:[{    "infrastructure_type": "road",    "damage": "damaged",    "location": "City"},{    "infrastructure_type": "airport",    "damage": "destroyed",    "location": "Airport"},{    "infrastructure_type": "railway",    "damage": "damaged",    "location": "Railway station"},{    "infrastructure_type": "harbor",    "damage": "flooded",    "location": "Harbor"},{    "infrastructure_type": "power grid",    "damage": "damaged",    "location": "Power plant"},{    "infrastructure_type": "water supply",    "damage": "contaminated",    "location": "Water treatment plant"},{    "infrastructure_type": "medical-care",    "damage": "destroyed",    "location": "Hospital"}]Note: The answers are based on the information provided in the context and the given infrastructure types, damage types, and locations. If there is no information provided in the context fo

KeyboardInterrupt: 

In [ ]:
df_responses_all[:]

,infrastructure_type,damage_type,location,filename,chunk
0,Klinik,Schwemmung,Innerhalb von Minuten,Korzilius 2021 Nach der Flut_cleaned.md,"Dann kommt die Flut. Innerhalb von Minuten wer­ den die Erdgeschosse in Klinik und Praxis über­ schwemmt, ein Feuerwehrauto und mehrere Rettungs­ wagen, mit denen Patienten aus dem Krankenhaus evakuiert werden sollen, laufen voll mit schlammigem Wasser. Feldgen flüchtet sich mit mehreren Kollegin­ nen und Kollegen aus dem Krankenhaus und einigen Patienten in die kleine Kapelle – sie liegt fünf Stufen höher als der Rest des Geländes. Doch auch dort dringt Wasser ein. Über eine Wendeltreppe retten sich alle in den ersten Stock des Nachbargebäudes, wo sie von den dort lebenden Ordensfrauen betreut werden, bis Ret­ tungskräfte mit schweren Lastwagen die vom Wasser Eingeschlossenen bergen können."
1,Praxis,Schwemmung,Innerhalb von Minuten,Korzilius 2021 Nach der Flut_cleaned.md,"Dann kommt die Flut. Innerhalb von Minuten wer­ den die Erdgeschosse in Klinik und Praxis über­ schwemmt, ein Feuerwehrauto und mehrere Rettungs­ wagen, mit denen Patienten aus dem Krankenhaus evakuiert werden sollen, laufen voll mit schlammigem Wasser. Feldgen flüchtet sich mit mehreren Kollegin­ nen und Kollegen aus dem Krankenhaus und einigen Patienten in die kleine Kapelle – sie liegt fünf Stufen höher als der Rest des Geländes. Doch auch dort dringt Wasser ein. Über eine Wendeltreppe retten sich alle in den ersten Stock des Nachbargebäudes, wo sie von den dort lebenden Ordensfrauen betreut werden, bis Ret­ tungskräfte mit schweren Lastwagen die vom Wasser Eingeschlossenen bergen können."
2,Krankenhaus,Schwemmung,Innerhalb von Minuten,Korzilius 2021 Nach der Flut_cleaned.md,"Dann kommt die Flut. Innerhalb von Minuten wer­ den die Erdgeschosse in Klinik und Praxis über­ schwemmt, ein Feuerwehrauto und mehrere Rettungs­ wagen, mit denen Patienten aus dem Krankenhaus evakuiert werden sollen, laufen voll mit schlammigem Wasser. Feldgen flüchtet sich mit mehreren Kollegin­ nen und Kollegen aus dem Krankenhaus und einigen Patienten in die kleine Kapelle – sie liegt fünf Stufen höher als der Rest des Geländes. Doch auch dort dringt Wasser ein. Über eine Wendeltreppe retten sich alle in den ersten Stock des Nachbargebäudes, wo sie von den dort lebenden Ordensfrauen betreut werden, bis Ret­ tungskräfte mit schweren Lastwagen die vom Wasser Eingeschlossenen bergen können."
3,Feldgen,Eingeschlossen,Zuhause,Korzilius 2021 Nach der Flut_cleaned.md,"Dann kommt die Flut. Innerhalb von Minuten wer­ den die Erdgeschosse in Klinik und Praxis über­ schwemmt, ein Feuerwehrauto und mehrere Rettungs­ wagen, mit denen Patienten aus dem Krankenhaus evakuiert werden sollen, laufen voll mit schlammigem Wasser. Feldgen flüchtet sich mit mehreren Kollegin­ nen und Kollegen aus dem Krankenhaus und einigen Patienten in die kleine Kapelle – sie liegt fünf Stufen höher als der Rest des Geländes. Doch auch dort dringt Wasser ein. Über eine Wendeltreppe retten sich alle in den ersten Stock des Nachbargebäudes, wo sie von den dort lebenden Ordensfrauen betreut werden, bis Ret­ tungskräfte mit schweren Lastwagen die vom Wasser Eingeschlossenen bergen können."
4,Kapelle,Eingeschossen,Zu Hause der Ordensfrauen,Korzilius 2021 Nach der Flut_cleaned.md,"Dann kommt die Flut. Innerhalb von Minuten wer­ den die Erdgeschosse in Klinik und Praxis über­ schwemmt, ein Feuerwehrauto und mehrere Rettungs­ wagen, mit denen Patienten aus dem Krankenhaus evakuiert werden sollen, laufen voll mit schlammigem Wasser. Feldgen flüchtet sich mit mehreren Kollegin­ nen und Kollegen aus dem Krankenhaus und einigen Patienten in die kleine Kapelle – sie liegt fünf Stufen höher als der Rest des Geländes. Doch auch dort dringt Wasser ein. Über eine Wendeltreppe retten sich alle in den ersten Stock des Nachbargebäudes, wo sie von den dort lebenden Ordensfrauen betreut werden, bis Ret­ tungskräfte mit schweren Lastwagen die vom Wasser Eingeschlossenen bergen können."
5,Nachbargebäu

For evaluating the ability of the LLM to extract the relevant information from the newspapers and scientific publication, we could either assess its performance manually by  comparing its single responses against the chunk text, or we could create our own validation dataset and compare the similarity between the LLM response and the respective validation sample. However, for now, we skip this evaluation part, and heading to a variation (and potential improvement) of previous workflow.

## Saving of the LLM responses
Save the LLM response as a csv file along with its corresponding prompt 

In [ ]:
LLM_OUTPUT_FILEPATH = LLM_OUTPUT_DIR / "llm_1_response.csv"


if not os.path.isfile(LLM_OUTPUT_DIR):

    print(f"Saving prompt and LLM response [.txt, .csv] to {LLM_OUTPUT_FILEPATH} ...")

    with open(f"prompt_{LLM_OUTPUT_DIR.stem}.txt", "w") as f:
        f.write(prompt_template.render(context=context, question=question))
    df_responses_all.to_csv(LLM_OUTPUT_FILEPATH, index=False)

else:
    print(f"Output file {Path(LLM_OUTPUT_FILEPATH).stem} already exists. Skip saving to avoid overwriting ...")




